# Deliverable 2
2026-06-21

Begin test assessments and complete Group 1 "required" tests.

| Stage | Stage specifications | Time required | Start-End |
| ----- | -------------------- | ------------- | --------- |
| 1 | Learn about the community and note shortcomings or additional toolboxes, such that the wheel need not be reinvented. | 3 weeks | May 1 - May 24 |
| **2** | **Begin test assessments and complete Group 1 “required” tests.** | **4 weeks** | **May 25 - June 21** |
| 3 | Complete Group 2 “Strongly Recommended” tests. | 2 weeks | June 22 - July 5 |
| 4 | Complete Group 3 “Suggested” tests. | 3 weeks | July 6 - July 26 |
| 5 | Explore other tests and finalize documentation.| 3 weeks | July 27 - August 16 |

Therefore, the objectives for this deliverable are to assess each of the following tests in both the QARTOD data manual and in the toolbox itself.

1. Timing/Gap Test
2. Syntax Test
3. Location Test
4. Gross Range Test
    * Including similar Climatological Test (strongly recommended)
    * "Seasonal expectations" variation on the gross range test.
    * Allen et al. 2012 - Variability Generalixed Digital Environmental Model
    * Boyer et al. 2013 - WOD
5. Pressure Test
    * Including similar Density Inversion Test (suggested)
    * Pressure is looking for a monotonically ordered record.
    * This is distinguished by checking potential density $\sigma _{\theta}$ increases with increasing depth. Easily done with `gsw`.
    * Could bundle this with test 8 - Rate of Change.

In [1]:
#   Imports of required packages/modules/libraries for use in this deliverable's tweaks.
#   For handling more generic types (see https://docs.python.org/3/library/stdtypes.html)
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from collections.abc import Sequence    #   list, tuple, range are the three basic sequence types.
    from numbers import Real                #   Real numbers, including `int` and `float` but not complex or imaginary numbers.

import numpy as np
import xarray as xr

In [2]:
#   Import a testing dataset
#   OG1 is OK?
fpath = "/home/aaron-mau/Data/OG1/delayed_SEA056_M102.nc"
ds = xr.load_dataset(fpath)

In [3]:
class QartodFlags:
    """Primary flags for QARTOD."""

    GOOD = 1
    UNKNOWN = 2
    SUSPECT = 3
    FAIL = 4
    MISSING = 9
FLAGS = QartodFlags  # Default name for all check modules
NOTEVAL_VALUE = QartodFlags.UNKNOWN

## Timing/Gap Test
* How is it described in the manual
* How is it represented in the code
* Does anything need to be done in either?

### Manual description of timing/gap test
It's a **required** test, so it's basically one of the most important checks that can be done on the data. Which makes sense - time is one of the dimensions of the data. If you don't know when the data was taken, it loses much of its value.

The manual describes it as "Check for arrival of data"
> Test determines that the most recent profile has been received within the expected time window
> (TIM_INC) and has the correct time stamp (TIM_STMP).
> 
> **Note:** For those gliders that do not update at regular intervals, a large value for TIM_STMP can be
> assigned. The gap check is not a solution for all timing errors. Data could be measured or received
> earlier than expected. This test does not address all clock drift/jump issues.

This is looking at expected time windows `(TIME_INC)` and has an appropriate time stamp `(TIM_STMP)`. In the example, `TIM_INC` is set to 6 hours.

Only 2 flags to assign: Either it passes the test (flag = 1) or it fails by not falling within the specified reporting range (flag = 4).

> `if NOW - TIM_STMP > TIM_INC, flag = 4`

In the manual, it isn't super clear what NOW is supposed to represent. It could be representing real-time applications, but in reality, it's probably just looking at the sequential data points. `NOW - TIM_STMP` is probably meant to be representing sequential data points, rather than whatever `TIM_STMP` value is supposed to be.

In this confusion, I had a discussion with one of my mentors. While it is doing two things, this doesn't apply to instances *not* in a real-time application. The fact that it suggests it is doing two tests in one could create compounding problems:
* The data has a good timestamp and is received in the designated time window (two successes, passes)
* The data has a good timestamp but isn't received in the designated time window (one success, fails)
* The data has a **bad** timestamp and isn't received in the designated time window (one success, fails)
* The data has a **bad** timestamp but **is** received within the time window (no successes, fails)

### Code for timing/gap test
I did a quick search for the manual terms, and there's already a disconnect. There is no term for `TIM_STMP` or `TIM_INC` in the entire `ioos_qc` package. I think this is worth bringing up, as users will probably investigate in the same way that I am now. Looking for the terminology that is described in the manual within the code to see how the parameters match up.

Furthermore, there is no defined function or class within `qartod.py` that describes something representing a timing or gap test.
* `utils.py` has a "check_timestamps" function. However, the docstring explicitly says "This is not a QARTOD test, but rather a utility test to make sure **times are in the proper order** and optionally **do not have large gaps prior to processing the data**. It takes in a `max_time_interval` value and returns a boolean. It does not return flags - rather a single boolean of pass/fail.

**Question for Filipe**: We want to reuse code as much as possible, as defined in the GSoC project proposal. Does it make sense to "graduate" this function, or are we sacrificing utility? This `utils.check_timestamps` function is called in `test_utils.py`, so I don't know how it is actually used (if at all).
* It also *adds* a function of confirming that the times are in the proper order. This may not be desireable or in the scope of this description in the manual. **Question for Filipe**: Is that itself worthy of an additional test?
* **A:** It's OK to redefine a new test or series of tests within QARTOD for simplicity. If the existing material in `utils` is not being used extensively, it possibly fills a use case that isn't intended for QARTOD. **A:** If you have additional functionalities you'd like to add, feel free to break them out and test them.

**Question for Filipe**: This is running on time, which is often a coordinate. Do you envision needing to do anything differently here? 
* **A:** No.

**Question for Filipe**: Is there a timestamp format that we should be expecting? Should it be homogenous in the notebook?
* **A:** A good way to get started is using the built-in numpy [datetime64 format](https://numpy.org/doc/stable/reference/arrays.datetime.html).

### Actions for timing/gap test
* (For external workgroup) I think that the manual's description of the test could be improved. "Check for arrival of data" $\rightarrow$ "Check data reporting period".
    * Filipe agrees. The QARTOD working group should go over what this test is intended for in the manual and consider a revision to prevent future confusion.
* Build a new series of tests that fits the description in the manual (without having compounding problems), and an additional test that checks for gaps *within the data itself*. This means:
    1. `impossible_date_test` (similar to GSTPP test 1.2 - timestamps shouldn't be prehistoric) (this should only be flags 1 and 4)
    2. `data_reception_test` (timestamps should be recent for NRT applications) (as per the manual, this is only flags 1 and 4)
    3. `time_gap_test` (designate a window of a certain size and flag points following said gap)

#### impossible_date_test()
Since this is my first change that I'll be proposing to the QARTOD QC, we'll want to preserve the existing codebase and make sure that everything is consistent. This means the docstrings at the beginning of the function should be similar to other existing tests, typing should be consistent, and the flag exports should be the same general item.

There are a few notes to make prior to starting this test, however.

##### Numpy masked arrays
Most of the other tests use numpy **masked** arrays, of which I wasn't particularly familiar with.

[Masked arrays](https://numpy.org/doc/stable/reference/maskedarray.generic.html) seem to be handy because they consist of the data and a mask. Kind of like a way to track a condition on each point in a data array.

In the documentation, the following example is given:
```python
import numpy as np
import numpy.ma as ma
y = ma.array([1, 2, 3], mask = [0, 1, 0])
```

In this case, the second point (index 1) is 1, meaning that this element is invalid. It's a great way to keep track of tests that are run on data (pass/fail, yes/no).

These arrays are commonly referenced using `y.data` or `y.mask` for the values and mask, respectively. Cool! I haven't worked very much with this before and it's much simpler than assigning multiple ndarrays to handle the same thing.

##### Regarding datetime64
[Numpy's datetime64](https://numpy.org/doc/stable/reference/arrays.datetime.html) is handy for a lot of things, but keep in mind that it is usually formatted as a timestamp *since Jan 1st, 1970*. The year, month, day, and minute can all be pulled out with some simple string formatting (like `"datetime64[Y]"), but you'll need to convert these items relative to 1970. *Note that you should convert it to an integer before using it.*
* datetime64[Y] $\rightarrow$ add '1970' to it.
* datetime64[M] $\rightarrow$ mod 12 and add 1 for the calendar month.
* datetime64[h] $\rightarrow$ mod 24 gives you the hour of the day.
* datetime64[m] $\rightarrow$ mod 60 gives you the minute of the hour.

For the dates in the month, you've got cases like leap years which we'll want to account for which adds a 29th day to February. We can get how long each month is using each calendar month, converting that to the day, and doing the same for the next month's start. Then the difference between them should be how long the month is, and which valid days there are.

`first day of March - first day of February = length of February`

Furthermore, we want the day of the month to be valid. We could approach this from a complicated series of if statements based on Julian date and the current year, calculating out each month independently, but we can also convert from months to days using Numpy's `astype` arguments.

In [4]:
def impossible_date_test(
        tinp: Sequence[Real],
        fail_span: tuple[Real, Real] | None = None,
) -> np.ma.core.MaskedArray:
    """
    This test confirms that the date and time for the data are reasonable.
    
    Given an array of time data, this test breaks the data down into a series of sub-tests.
    These tests are similar to those outlined in test 1.2 of the GTSPP RTQC Manual (IOC, 2010).
    * The year is either present or in the past.
    * The month is within 1-12.
    * The day is possible for the given month, depending on year.
    * The hour of the day is between 0-23.
    * The minutes are between 0-50.
    * (Optional) The time data is within a user-defined tolerance.
    Data deemed to have failed any or all of these tests are flagged as FAIL. Any missing and masked data is flagged as UNKNOWN.

    Parameters
    ----------
    tinp
        Time input data as a numeric numpy array or list of real numbers.
    fail_span
        2-tuple range which to flag outside data as FAIL. [optional]

    Returns
    -------
    flag_arr
        A masked array of flag values equal in size to that of the input `tinp`.
    """
    #   Init 
    original_shape = tinp.shape
    tinp = np.ma.asarray(tinp, dtype="datetime64[ns]").flatten()
    flag_arr = np.ma.ones(tinp.size, dtype="uint8") #   Init to flag 1 "good"
    
    tinp.mask = np.isnat(tinp.data)
    flag_arr[tinp.mask] = QartodFlags.MISSING   #   Init missing timestamps to the missing flag
    valid = ~tinp.mask  #   Define where the data point are not missing, such that we can index them and keep those flags.

    #   Year test - where year is in the future
    yr = tinp.data.astype("datetime64[Y]").astype(int) + 1970
    current_yr = np.datetime64("now", "Y").astype(int) + 1970
    flag_arr[valid & (yr > current_yr)] = QartodFlags.FAIL

    # #   Month test - the month must be 1-12
    # mn = tinp.data.astype("datetime64[M]").astype(int) % 12 + 1
    # flag_arr[valid & ((mn < 1) | (mn > 12))] = QartodFlags.FAIL
    
    # #   Day test - day must be possible for the given month
    # #   Define the start of the first and next month for all points in the array
    # mn_1 = tinp.data.astype("datetime64[M]").astype("datetime64[D]")
    # mn_2 = (tinp.data.astype("datetime64[M]") + 1).astype("datetime64[D]")
    # days = (mn_2 - mn_1).astype(int)
    # #   Day of the month
    # dy   = (tinp.data.astype("datetime64[D]") - mn_1).astype(int) + 1
    # flag_arr[valid & ((dy < 1) | (dy > days))] = QartodFlags.FAIL

    # #   Hour test - must be 0-23
    # hr = tinp.data.astype("datetime64[h]").astype(int) % 24
    # flag_arr[valid & ((hr < 0) | (hr > 23))] = QartodFlags.FAIL

    # #   Minute test - must be 0-59
    # mi = tinp.data.astype("datetime64[m]").astype(int) % 60
    # flag_arr[valid & ((mi < 0) | (mi > 59))] = QartodFlags.FAIL

    #   Span test
    if fail_span is not None:
        #   Convert if not already
        low, high = np.datetime64(fail_span[0]), np.datetime64(fail_span[1])
        flag_arr[valid & ((tinp.data < low) | (tinp.data > high))] = QartodFlags.FAIL
    
    return flag_arr.reshape(original_shape)

Discovery while testing: Numpy datetime64 doesn't actually allow months, days, hours, or minutes to be outside of tolerated ranges.

Inserting a fake datetime throws a ValueError:
```
np.datetime64("2025-12-32")
ValueError: Day out of range in datetime string "2025-12-32"

np.datetime64("2026-01-12T17:61:07.000000000")
ValueError: Minutes out of range in datetime string "2026-01-12T17:61:07.000000000"
```
As such, we can comment out many of the checks we defined. It even handles leap days.

```
np.datetime64("2024-02-29") #   This was a leap day

np.datetime64("2025-02-29") #   Most certainly not a leap day
ValueError: Day out of range in datetime string "2025-02-29"
```
The minutia described in the GTSPP is handled by datetime64 - if there are impossible values with the date and time then it will error out before getting tested and that tends to be easy to find.

Therefore, the most important aspect of this is probably the span test and checking for the future year. This can be further simplified by just looking for dates in the future, so we won't need to break out the year at all.

The final function to be added should then be

In [5]:
def impossible_date_test(
        tinp: Sequence[Real],
        fail_span: tuple[Real, Real] | None = None,
) -> np.ma.core.MaskedArray:
    """
    This test confirms that the date and time for the data are reasonable.
    
    Given an array of time data, this test breaks the data down into a series of sub-tests.
    These tests are similar to those outlined in test 1.2 of the GTSPP RTQC Manual (IOC, 2010).
    * The year is either present or in the past.
    * (Optional) The time data is within a user-defined tolerance.
    Data deemed to have failed any or all of these tests are flagged as FAIL. Any missing and masked data is flagged as UNKNOWN.

    Parameters
    ----------
    tinp
        Time input data as a numeric numpy array or list of real numbers.
    fail_span
        2-tuple range which to flag outside data as FAIL. [optional]

    Returns
    -------
    flag_arr
        A masked array of flag values equal in size to that of the input `tinp`.
    """
    #   Init 
    original_shape = tinp.shape
    tinp = np.ma.asarray(tinp, dtype="datetime64[ns]").flatten()
    flag_arr = np.ma.ones(tinp.size, dtype="uint8") #   Init to flag 1 "good"
    
    tinp.mask = np.isnat(tinp.data)
    flag_arr[tinp.mask] = QartodFlags.MISSING   #   Init missing timestamps to the missing flag
    valid = ~tinp.mask  #   Define where the data point are not missing, such that we can index them and keep those flags.

    #   Check for time travelers
    now = np.datetime64("now")
    flag_arr[valid & (tinp.data > now)] = QartodFlags.FAIL

    #   Span test
    if fail_span is not None:
        #   Convert if not already
        low, high = np.datetime64(fail_span[0]), np.datetime64(fail_span[1])
        flag_arr[valid & ((tinp.data < low) | (tinp.data > high))] = QartodFlags.FAIL
    
    return flag_arr.reshape(original_shape)

Let's confirm that the checker works. Primarily, this is confirming that there are no NaT values, future dates, or anything outside of our span.

In [6]:
flags = impossible_date_test(ds.TIME.to_numpy(), fail_span = ("2020-01-01T00:00:00.0000", "2027-01-01T00:00:00.0000"))
print(any(flags == QartodFlags.FAIL))
print(any(flags == QartodFlags.MISSING))

False
False


Let's insert a bad data point in the future and confirm that we get one bad point out.

In [7]:
bad_data = ds.TIME.to_numpy().copy()
bad_data[20000] = np.datetime64("2088-01-12T23:05:16.000000000")    #   (Cheers to the QARTOD users in 2088)
print(bad_data[19998:20002])

['2026-01-12T23:05:14.000000000' '2026-01-12T23:05:15.000000000'
 '2088-01-12T23:05:16.000000000' '2026-01-12T23:05:17.000000000']


In [8]:
flags = impossible_date_test(bad_data)
print(any(flags == QartodFlags.FAIL))
print(np.where(flags == QartodFlags.FAIL)[0][0])    #   Return the index - np.where nests things in a tuple of arrays

True
20000


And we'll modify the data further by inserting a chunk of dates that are outside of our `fail_span`. We'll insert a lot, so let's just return the first and last 5 indexes where the flags are bad. We should still see the bad time from before at index 20000.

In [9]:
bad_data[1500:2200] = np.datetime64("2013-01-12T23:05:16.000000000")
flags = impossible_date_test(bad_data, fail_span=("2020-01-01T00:00:00.0000","2027-01-01T00:00:00.0000"))
idx = np.where(flags == QartodFlags.FAIL)[0]
print(idx[:5],idx[-5:])

[1500 1501 1502 1503 1504] [ 2196  2197  2198  2199 20000]


##### Another finding - duplicate timestamps
I was reading through the Rutgers glider QC and found a `check_duplicate_timestamps.py` file ([link here](https://github.com/gliderqctest/ruglider_qc/blob/master/check_duplicate_timestamps.py)), which looks for duplicate timestamps across multiple files. Though multiple files is probably out of the scope of QARTOD, we should be able to support flagging multiple identical timestamps in a file. There are examples of [xarray instances where this can occur](https://stackoverflow.com/questions/51058379/drop-duplicate-times-in-xarray), and I can see it causing problems later on when we differentiate the items in our arrays.

Not quite clear what we should do in this instance or where we should put this.

**Question for Filipe:** Where would you put a `duplicate_timestamp` test, and what would it do?
* Allow it to error out, or raise an exception.
* Print the warning to the log/console for the user to know that there is an issue.
* Flag one or all duplicate times as suspect (3) or fail (4).

#### data_reception_test()
This is a pretty straightforward test that should be more useful to datacenters that are expecting data within a certain timeframe. In many ways, it seems like a subset of the previous test. In the manual, the time window is described as `TIM_INC`, where the example sets it to 6 hours. In this case, our `fail_span` can be a single value, in hours, which if the data is older than by that amount, will cause it to be flagged.

It might be helpful to define a `starting timestamp` value of some sort (I'll call it `from_time`), such that we can check if data were received within X hours of timestamp Y. This isn't explicitly stated in the manual, so let's have it be optional and default to `np.datetime64("now")`.

In [10]:
def data_reception_test(
        tinp: Sequence[Real],
        fail_span: Real = 6,
        from_time: Real | None = None,
) -> np.ma.core.MaskedArray:
    """
    This test checks for data timestamps to be within a certain amount of time of present.
    
    Timestamps that are further away in time from the `from_time` variable than that designated
    by `fail_span` are flagged as FAIL. Data points that are newer than `from_time` are not considered
    for this test and are passed through as GOOD. Any missing and masked data is flagged as UNKNOWN.

    Parameters
    ----------
    tinp
        Time input data as a numeric numpy array or list of real numbers.
    fail_span
        A numeric value in hours which, if data is older than, is flagged as FAIL.
    from_time
        A timestamp which, if defined, is used to anchor measurements against. Defaults to current date and time. [optional]

    Returns
    -------
    flag_arr
        A masked array of flag values equal in size to that of the input `tinp`.
    """
    #   Init 
    if from_time == None:
        from_time = np.datetime64("now")
    else:
        from_time = np.datetime64(from_time)
    
    original_shape = tinp.shape
    tinp = np.ma.asarray(tinp, dtype="datetime64[ns]").flatten()
    flag_arr = np.ma.ones(tinp.size, dtype="uint8") #   Init to flag 1 "good"
    
    tinp.mask = np.isnat(tinp.data)
    flag_arr[tinp.mask] = QartodFlags.MISSING   #   Init missing timestamps to the missing flag
    valid = ~tinp.mask  #   Define where the data point are not missing, such that we can index them and keep those flags.
    
    fail_span = np.timedelta64(int(fail_span * 3600), "s")
    diff_time = from_time - tinp.data
    flag_arr[valid & (diff_time > fail_span)] = QartodFlags.FAIL
    
    return flag_arr.reshape(original_shape) 

Let's test it with an arbitrary value inside the dataset, using the default 6 hour tolerance for flagging. Everything after this 6 hour tolerance should be flagged as 1, whereas everything before it should be flagged as 4.

In [11]:
from_time = np.datetime64('2026-02-10T11:17:07.000000000')
print(np.where(ds.TIME == from_time))

(array([1690309]),)


In [12]:
flags = data_reception_test(ds.TIME.to_numpy(), from_time=from_time)
print(f"The last index where data are flagged: {np.where(flags == 4)[0][-1]}")
print(f"Length of the array: {len(ds.TIME)}")

The last index where data are flagged: 1669201
Length of the array: 1690311


In [13]:
print(flags[1669195:1669210])   #   Looks to be index 1669201 is just over 6 hours away.
print(np.datetime64(from_time - ds.TIME[1669202].item()))   #   Returns just 6 hours from Jan 1, 1970

[4 4 4 4 4 4 4 1 1 1 1 1 1 1 1]
1970-01-01T06:00:00.000000000


#### time_gap_test()
This is the first test that probably *shouldn't* be used on NRT data, as it looks at the gaps within the data itself. Similar to the previous test, given a threshold, it takes the diff of the time array and flags the point *after* any gaps that are exceeding the specified `fail_span`.

Since we'll take the difference of the values against each other, the final array will be of size N-1, presenting a [fencepost](https://en.wikipedia.org/wiki/Off-by-one_error#Fencepost_error) problem. This untested point can be left as UNTESTED or UNKNOWN, unless we work backwards and confirm that it is less than `fail_span` from the next point.

In [14]:
def time_gap_test(
    tinp: Sequence[Real],
    fail_span: Real = 2,
) -> np.ma.core.MaskedArray:
    """
    This test checks for gaps in the data time that exceed a specific threshold.
    
    The data `tinp` is differentiated and those changes in time are compared against the time threshold
    `fail_span`. If the difference in time between points exceeds the value of `fail_span`, the following
    data point is flagged as FAIL. The first point inherits the flag of the second point, unless the
    second point is UNKNOWN or MISSING. Any missing and masked data is flagged as UNKNOWN.

    Parameters
    ----------
    tinp
        Time input data as a numeric numpy array or list of real numbers.
    fail_span
        A numeric value for time duration in hours which, if timestamps jump by more than between points, flags following data point as FAIL.

    Returns
    -------
    flag_arr
        A masked array of flag values equal in size to that of the input `tinp`.
    """
    
    original_shape = tinp.shape
    tinp = np.ma.asarray(tinp, dtype="datetime64[ns]").flatten()
    flag_arr = np.ma.ones(tinp.size, dtype="uint8") #   Init to flag 1 "good"
    
    tinp.mask = np.isnat(tinp.data)
    flag_arr[tinp.mask] = QartodFlags.MISSING   #   Init missing timestamps to the missing flag
    valid = ~tinp.mask

    diff_time = np.ma.zeros(tinp.size, dtype="timedelta64[ns]") #   Guarantee that the array is the same size
    diff_time[1:] = tinp.data[1:] - tinp.data[:-1]

    #   NaT testing - guarantee that the previous entry isn't NaT
    valid[1:] &= valid[:-1]

    fail_span = np.timedelta64(int(fail_span * 3600), "s")
    flag_arr[valid & (diff_time > fail_span)] = QartodFlags.FAIL

    #   Handle the first point
    if tinp.mask[1]:
        flag_arr[0] = QartodFlags.UNKNOWN
    else:
        flag_arr[0] = flag_arr[1]

    return flag_arr.reshape(original_shape)

In [15]:
flags = time_gap_test(ds.TIME)
print(any(flags == 4))
print(np.where(flags == 4))

True
(array([1414125, 1449545, 1449929, 1476935, 1512225, 1630895]),)


This is the first test I've run so far where there are actual results on the data I've been testing. In this case, I've got 6 potential gaps in the data (ending at the returned indices above). Let's investigate them a little further.

In [16]:
print(np.diff(ds.TIME[1414122:1414130]))    #   Difference in nanoseconds [ns] ix = 2
print(ds.TIME[1414122:1414130])

[    1000000000     1000000000 76010000000000     1000000000
     1000000000     1000000000     1000000000]
<xarray.DataArray 'TIME' (N_MEASUREMENTS: 8)> Size: 64B
array(['2026-01-29T11:25:48.000000000', '2026-01-29T11:25:49.000000000',
       '2026-01-29T11:25:50.000000000', '2026-01-30T08:32:40.000000000',
       '2026-01-30T08:32:41.000000000', '2026-01-30T08:32:42.000000000',
       '2026-01-30T08:32:43.000000000', '2026-01-30T08:32:44.000000000'],
      dtype='datetime64[ns]')
Coordinates:
  * N_MEASUREMENTS  (N_MEASUREMENTS) int64 64B 1414122 1414123 ... 1414129
    LATITUDE        (N_MEASUREMENTS) float64 64B 55.45 55.45 ... 55.43 55.43
    LONGITUDE       (N_MEASUREMENTS) float64 64B 16.23 16.23 ... 16.28 16.28
    TIME            (N_MEASUREMENTS) datetime64[ns] 64B 2026-01-29T11:25:48 ....
    DEPTH           (N_MEASUREMENTS) float64 64B 1.365 1.293 ... 1.271 1.264
Attributes:
    interpolation_methodology:  
    long_name:                  Time elapsed since 1970-01-01T00:00:

In [17]:
gap = np.diff(ds.TIME[1414122:1414130])[2]
print(gap.astype("timedelta64[h]"))

21 hours


Sure enough, this data has a gap in it of about 21 hours in length. And given that we found it using the flag, we should be good to ID other points with the same issue using the flags.

In [18]:
idx = np.where(flags == 4)[0]
gaps = (ds.TIME.values[idx] - ds.TIME.values[idx - 1]).astype("timedelta64[h]")
print(gaps)
print(ds.TIME.values[idx])

[ 21  20   4 128  13  21]
['2026-01-30T08:32:40.000000000' '2026-01-31T15:30:16.000000000'
 '2026-01-31T20:07:25.000000000' '2026-02-06T12:28:44.000000000'
 '2026-02-07T11:37:37.000000000' '2026-02-09T18:23:44.000000000']


So we can see that there are data gaps of variable length present within the data, including a very long one of 128 hours preceeding February 6th.

There is one last thing to test - to confirm that the first data point is flagged correctly.

In [19]:
bad_data = ds.TIME.copy()
bad_data[1] = np.nan

In [20]:
flags = time_gap_test(bad_data[0:10])
print(flags)

[2 9 1 1 1 1 1 1 1 1]


#### Unit testing
Now! To take these tests that we've done on each of the timing/gap tests and get them incorporated into the **unit** tests.

Unit tests are helpful because they:
* Confirm that the code does what we expect it to do in a controlled environment.
* To a certain extent, documentation. If your tests are well thought-out, users will almost have examples of the code in the tests.
* Demonstrate robustness in the code, should Python functions update or if you [refactor anything](https://en.wikipedia.org/wiki/Code_refactoring).
* Finding bugs.

There are loads of quotes out there about how good unit tests make for good, reliable code. When adding functionality, it's very important to add or modify unit tests to confirm that the code still works the way you intend it to.

All of these unit tests are typically done in Python using `pytest` or `unittest`. In this instance, I test using `pytest` with a tweaked output.

Navigating to the `ioos_qc` **/tests/** folder, `pytest` is pretty straightforward to call.

```shell
pytest test_qartod.py

============================================================ test session starts =============================================================
platform linux -- Python 3.14.4, pytest-9.0.3, pluggy-1.6.0 -- /home/aaron-mau/miniforge3/envs/gsoc/bin/python3.14
cachedir: .pytest_cache
rootdir: /home/aaron-mau/Code/gsoc/mau/ioos_qc
configfile: pyproject.toml
plugins: anyio-4.13.0, cov-7.1.0
collected 69 items                                                                                                                           

test_qartod.py::QartodLocationTest::test_location PASSED
test_qartod.py::QartodLocationTest::test_location_bad_input PASSED
test_qartod.py::QartodLocationTest::test_location_bbox PASSED
test_qartod.py::QartodLocationTest::test_location_distance_threshold PASSED
test_qartod.py::QartodLocationTest::test_single_location_nan PASSED
test_qartod.py::QartodLocationTest::test_single_location_none PASSED
test_qartod.py::QartodGrossRangeTest::test_gross_range_bad_input PASSED
test_qartod.py::QartodGrossRangeTest::test_gross_range_check PASSED
test_qartod.py::QartodGrossRangeTest::test_gross_range_check_masked PASSED
test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_day_of_year PASSED
test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_monthly PASSED
test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_quarter PASSED
test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_week_of_year PASSED
test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_dayofyear_periods PASSED
test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_monthly_periods PASSED
test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_quarterly_periods PASSED
test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_weekofyear_periods PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_maximum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_minimum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_out_of_range_high PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_out_of_range_low PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_maximum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_minimum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_out_of_range_high PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_out_of_range_low PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_maximum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_minimum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_out_of_range_high PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_out_of_range_low PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_maximum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_minimum PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_out_of_range_high PASSED
test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_out_of_range_low PASSED
test_qartod.py::QartodClimatologyDepthTest::test_climatology_test_all_unknown PASSED
test_qartod.py::QartodClimatologyMissingTest::test_climatology_missing_values PASSED
test_qartod.py::QartodClimatologyTest::test_climatology_test PASSED
test_qartod.py::QartodClimatologyTest::test_climatology_test_depths PASSED
test_qartod.py::QartodClimatologyTest::test_climatology_test_fail PASSED
test_qartod.py::QartodClimatologyTest::test_climatology_test_seconds_since_epoch PASSED
test_qartod.py::QartodSpikeTest::test_spike PASSED
test_qartod.py::QartodSpikeTest::test_spike_initial_final_values PASSED
test_qartod.py::QartodSpikeTest::test_spike_masked PASSED
test_qartod.py::QartodSpikeTest::test_spike_methods PASSED
test_qartod.py::QartodSpikeTest::test_spike_negative_vals PASSED
test_qartod.py::QartodSpikeTest::test_spike_realdata PASSED
test_qartod.py::QartodSpikeTest::test_spike_test_bad_method PASSED
test_qartod.py::QartodSpikeTest::test_spike_test_inputs PASSED
test_qartod.py::QartodRateOfChangeTest::test_rate_of_change PASSED
test_qartod.py::QartodRateOfChangeTest::test_rate_of_change_fail_flag PASSED
test_qartod.py::QartodRateOfChangeTest::test_rate_of_change_missing_values PASSED
test_qartod.py::QartodRateOfChangeTest::test_rate_of_change_negative_values PASSED
test_qartod.py::QartodFlatLineTest::test_flat_line PASSED
test_qartod.py::QartodFlatLineTest::test_flat_line_missing_values PASSED
test_qartod.py::QartodFlatLineTest::test_flat_line_short_timeseries PASSED
test_qartod.py::QartodFlatLineTest::test_flat_line_starting_from_beginning PASSED
test_qartod.py::QartodFlatLineTest::test_flat_line_with_spike PASSED
test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal PASSED
test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_missing PASSED
test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_missing_time_window PASSED
test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_range PASSED
test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_time_window PASSED
test_qartod.py::QartodDensityInversionTest::test_density_inversion_bad_depth_value PASSED
test_qartod.py::QartodDensityInversionTest::test_density_inversion_down_up_cast_flags PASSED
test_qartod.py::QartodDensityInversionTest::test_density_inversion_downcast_flags PASSED
test_qartod.py::QartodDensityInversionTest::test_density_inversion_input PASSED
test_qartod.py::QartodDensityInversionTest::test_density_inversion_one_record_input PASSED
test_qartod.py::QartodDensityInversionTest::test_density_inversion_stable_depth_flags PASSED
test_qartod.py::QartodDensityInversionTest::test_density_inversion_upcast_flags PASSED
test_qartod.py::QartodUtilsTests::test_qartod_compare PASSED

============================================================= 69 passed in 1.10s =============================================================
```

This lists each of the tests that are accounted for in the `test_qartod.py` code. In this case, there are 69 tests, consisting of various **assertions** (basically a confirmation that the code has done what we're expecting).

This is nice, but we can add to this output by using `cov`, a [method of tracking](https://about.codecov.io/) which lines of code have been tested. For example, each of the tests listed above can be passed, but it's possible that not 100% of the lines in the code are being tested. This is great for finding those edge scenarios, like with lots of `if` statements or data conditions.

I modify the line for `pytest` with additional `cov` arguments:
```shell
pytest test_qartod.py --cov-report=term-missing --cov=ioos_qc

... (the same pytest output as above) ...

______________________________________________ coverage: platform linux, python 3.14.4-final-0 _______________________________________________

Name                                                                             Stmts   Miss  Cover   Missing
--------------------------------------------------------------------------------------------------------------
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/__init__.py                            4      2    50%   5-6
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/_version.py                            1      0   100%
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/argo.py                               51     51     0%   3-154
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/axds.py                               43     43     0%   3-120
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/config.py                            235    235     0%   12-549
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/config_creator/__init__.py             2      2     0%   1-9
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/config_creator/config_creator.py     219    219     0%   1-633
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/config_creator/fx_parser.py           66     66     0%   25-165
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/config_creator/get_assets.py         140    140     0%   3-288
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/config_creator/make_config.py         11     11     0%   1-34
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/gliders.py                             5      5     0%   3-14
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/plotting.py                           60     60     0%   1-182
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/qartod.py                            346     44    87%   61-62, 83-84, 143-144, 153-154, 219-220, 238-239, 302, 311-328, 339-340, 346-347, 352-353, 358-359, 366-368, 456-459, 875-876, 896, 960-961, 968
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/results.py                            87     87     0%   1-181
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/stores.py                            180    180     0%   1-427
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/streams.py                           247    247     0%   1-592
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/utils.py                             141     83    41%   40-41, 46-100, 111-148, 178, 181, 194-196, 220-226, 236-245, 251-253, 258-265, 273-282
--------------------------------------------------------------------------------------------------------------
TOTAL                                                                             1838   1475    20%
```

This covers all of the individual tests in the folder (even though I specified `test_qartod.py`, not sure why that went awry). As you can see, `qartod.py` has about 87% coverage before adding these new methods (and the average for the whole `ioos_qc` toolbox is 20%, but that's not our priority). The specific lines that are untested are given in the "Missing" column. When we add additional methods, we can expect this coverage to drop.

After adding these new methods to the end, we can see that, indeed, coverage drops becuase there are 3 fat regions of code that haven't been tested after line 1024.

```shell
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/qartod.py                            389     84    78%   61-62, 83-84, 143-144, 153-154, 219-220, 238-239, 302, 311-328, 339-340, 346-347, 352-353, 358-359, 366-368, 456-459, 875-876, 896, 960-961, 968, 1024-1042, 1072-1089, 1117-1140
```

In exploring the existing unit tests, it looks like we're using a Class system. As an example:

```python
class QartodSpikeTest(unittest.TestCase):
    def setUp(self):
        self.suspect_threshold = 25
        self.fail_threshold = 50

    def test_spike(self):
        """Test to make ensure single value spike detection works properly."""
        arr = [10, 12, 999.99, 13, 15, 40, 9, 9]

        # First and last elements should always be good data, unless someone
        # has set a threshold to zero.
        expected = [2, 4, 4, 4, 1, 3, 1, 2]

        inputs = [
            arr,
            np.asarray(arr, dtype=np.float64),
            dask_arr(np.asarray(arr, dtype=np.float64)),
        ]
        for i in inputs:
            npt.assert_array_equal(
                qartod.spike_test(
                    inp=i,
                    suspect_threshold=self.suspect_threshold,
                    fail_threshold=self.fail_threshold,
                ),
                expected,
            )
#   much more below
```
This is different from how I've worked on unit tests before, but I can see why this would be helpful.
* Some of the `qartod` methods, like the climatology test, use Classes already. In some ways, this may help mimic their functionality.
* It means you don't need to use *decorators* or [*fixtures*](https://docs.pytest.org/en/stable/how-to/fixtures.html), which allow you to pass fake data around. Here, you can define your data when you initialize the testing class, allowing you to then pass that along to all the follow up tests.
* The alternative, which I've done, is just have lots and lots of individually-named functions. Like the `test_spike` example above, but for each and every scenario I could imagine to get 100% coverage of the original code. Having them all live within a class helps keep things organized.

So let's get started! First thing's first, I've used some data in testing that I could probably reduce in size and copy into my class when it's initialized. This would be `ds.TIME` or `bad_data`, good for testing the different scenarios on each of the new methods we've added. All tests within each class will need to have the "`test_`" prefix.
* impossible_date_test()
    * For this one, we're very reliant on `np.datetime64` handling errors with impossible dates or times. In testing, we should get it to error out. This is what `unittest.TestCase` does for us elsewhere in the code.
* data_reception_test()
* time_gap_test()

In [21]:
import unittest
from ioos_qc import qartod  #   Since I've added the functions to my local version of qartod, my new functions should show up

In [22]:
"impossible_date_test" in dir(qartod)   #   dir(object) lists the things you can do with it. A good way to confirm we're ready to test!

True

In typing up these tests, it becomes clearer that `shape` is a limiting factor on some of my data. At first, I tried
```python
dt = "2026-13-91T00:00:00.000"
```
but neither that nor `list` have the `shape` attribute, so it broke in testing. As such, I have to move forward with a numpy array.
```python
dt = np.array("2026-13-91T00:00:00.000")
```

In [23]:
class QartodImpossibleDateTest(unittest.TestCase):
    def setUp(self):
        """Define the data that we're going to pass into the following tests.
        
        Within the class, these data will live as attributes of 'self'."""
        #   As we defined them before.
        times = ['2026-01-12T23:05:14.000000000','2026-01-12T23:05:15.000000000','2026-01-12T23:05:16.000000000','2026-01-12T23:05:17.000000000']
        self.data_good = np.array(times, dtype="datetime64")
        self.data_bad = self.data_good.copy()
        self.data_bad[1] = np.datetime64("2088-01-12T23:05:16.000000000")
    def test_error_bad_datetime(self):
        dt = np.array("2026-13-91T00:00:00.000")    #   the function will attempt to convet this to datetime64
        self.assertRaises(
            ValueError,
            qartod.impossible_date_test,
            tinp=dt,
        )   #   This is a type of assertion that checks for errors
        dt = np.array("2026-12-00T38:00:00.000")
        self.assertRaises(
            ValueError,
            qartod.impossible_date_test,
            tinp = dt,
        )
    def test_all_nat(self):
        dt = np.full(4, np.datetime64("NaT"))
        flags = qartod.impossible_date_test(tinp = dt)
        assert all(flags == 9)
    def test_future_year(self):
        """Uses data_bad, which has an entry set well in the future."""
        flags = qartod.impossible_date_test(tinp = self.data_bad)
        assert 4 in flags   #   This is a more typical assertion. Here, we expect there to be a bad entry in the flags.
        assert flags[1] == 4
        assert type(flags) == np.ma.core.MaskedArray
    def test_span_good(self):
        flags = qartod.impossible_date_test(tinp = self.data_good, fail_span=("2012-01-01T00:00:00.000", "2027-01-01T00:00:00.000"))
        assert all(flags == 1)


So now that we've made our assertions, we should be able to copy this over to `qartod.py` and try running our unit testing on it.

If I recall correctly, this can be done inside a notebook. Let's confirm that this looks right.

In [24]:
import pytest
testing_path = "/home/aaron-mau/Code/gsoc/mau/ioos_qc/tests/test_qartod.py"
# pytest.main([full])   #   Learning how to run pytest in a jupyter notebook...
pytest.main(["--assert=plain", "-v", testing_path])

============================= test session starts ==============================
platform linux -- Python 3.14.4, pytest-9.0.3, pluggy-1.6.0 -- /home/aaron-mau/miniforge3/envs/gsoc/bin/python
cachedir: .pytest_cache
rootdir: /home/aaron-mau/Code/gsoc/mau/ioos_qc
configfile: pyproject.toml
plugins: anyio-4.13.0, cov-7.1.0
collecting ... collected 73 items

../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_bad_input PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_bbox PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_distance_threshold PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_single_location_nan PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_single_location_none PASSED
../ioos_qc/tests/test_qartod.py::QartodGrossRangeTest::test_gross_range_bad_input PASSED
../ioos_qc/tests/test_qartod.py::QartodGrossRangeTes

<ExitCode.OK: 0>

Nice! It looks like it ran, and now it lists 73 tests, 4 more than before. That makes sense, as we now see QartodImpossibleDateTest at the bottom of it. Checking our code coverage, we should be able to add the `cov` options from before.

(As I will be adding more tests in the future, this coverage will continue to change. I'll insert a snapshot of what this looks like below. Probably fine to minimize this or the previous cell in the future.)

```
[1m============================= test session starts ==============================[0m
platform linux -- Python 3.14.4, pytest-9.0.3, pluggy-1.6.0 -- /home/aaron-mau/miniforge3/envs/gsoc/bin/python
cachedir: .pytest_cache
rootdir: /home/aaron-mau/Code/gsoc/mau/ioos_qc
configfile: pyproject.toml
plugins: anyio-4.13.0, cov-7.1.0
[1mcollecting ... [0mcollected 73 items

../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_bad_input [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_bbox [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_distance_threshold [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_single_location_nan [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_single_location_none [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodGrossRangeTest::test_gross_range_bad_input [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodGrossRangeTest::test_gross_range_check [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodGrossRangeTest::test_gross_range_check_masked [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_day_of_year [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_monthly [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_quarter [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodTest::test_climatology_test_periods_week_of_year [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_dayofyear_periods [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_monthly_periods [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_quarterly_periods [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyPeriodFullCoverageTest::test_weekofyear_periods [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_maximum [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_minimum [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_out_of_range_high [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_fspan_out_of_range_low [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_maximum [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_minimum [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_out_of_range_high [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_tspan_out_of_range_low [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_maximum [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_minimum [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_out_of_range_high [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_vspan_out_of_range_low [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_maximum [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_minimum [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_out_of_range_high [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyInclusiveRangesTest::test_zspan_out_of_range_low [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyDepthTest::test_climatology_test_all_unknown [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyMissingTest::test_climatology_missing_values [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyTest::test_climatology_test [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyTest::test_climatology_test_depths [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyTest::test_climatology_test_fail [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodClimatologyTest::test_climatology_test_seconds_since_epoch [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike_initial_final_values [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike_masked [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike_methods [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike_negative_vals [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike_realdata [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike_test_bad_method [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodSpikeTest::test_spike_test_inputs [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodRateOfChangeTest::test_rate_of_change [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodRateOfChangeTest::test_rate_of_change_fail_flag [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodRateOfChangeTest::test_rate_of_change_missing_values [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodRateOfChangeTest::test_rate_of_change_negative_values [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodFlatLineTest::test_flat_line [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodFlatLineTest::test_flat_line_missing_values [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodFlatLineTest::test_flat_line_short_timeseries [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodFlatLineTest::test_flat_line_starting_from_beginning [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodFlatLineTest::test_flat_line_with_spike [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_missing [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_missing_time_window [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_range [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodAttenuatedSignalTest::test_attenuated_signal_time_window [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodDensityInversionTest::test_density_inversion_bad_depth_value [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodDensityInversionTest::test_density_inversion_down_up_cast_flags [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodDensityInversionTest::test_density_inversion_downcast_flags [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodDensityInversionTest::test_density_inversion_input [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodDensityInversionTest::test_density_inversion_one_record_input [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodDensityInversionTest::test_density_inversion_stable_depth_flags [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodDensityInversionTest::test_density_inversion_upcast_flags [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodUtilsTests::test_qartod_compare [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodImpossibleDateTest::test_all_nat [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodImpossibleDateTest::test_error_bad_datetime [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodImpossibleDateTest::test_future_year [32mPASSED[0m
../ioos_qc/tests/test_qartod.py::QartodImpossibleDateTest::test_span_good [32mPASSED[0m

[32m============================== [32m[1m73 passed[0m[32m in 0.32s[0m[32m ==============================[0m
```

In [25]:
pytest.main([
    "--assert=plain", "-v",
    "--cov-report=term-missing",
    "--cov=ioos_qc",
    testing_path,
])

============================= test session starts ==============================
platform linux -- Python 3.14.4, pytest-9.0.3, pluggy-1.6.0 -- /home/aaron-mau/miniforge3/envs/gsoc/bin/python
cachedir: .pytest_cache
rootdir: /home/aaron-mau/Code/gsoc/mau/ioos_qc
configfile: pyproject.toml
plugins: anyio-4.13.0, cov-7.1.0
collecting ... collected 73 items

../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_bad_input PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_bbox PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_location_distance_threshold PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_single_location_nan PASSED
../ioos_qc/tests/test_qartod.py::QartodLocationTest::test_single_location_none PASSED
../ioos_qc/tests/test_qartod.py::QartodGrossRangeTest::test_gross_range_bad_input PASSED
../ioos_qc/tests/test_qartod.py::QartodGrossRangeTes

/home/aaron-mau/miniforge3/envs/gsoc/lib/python3.14/site-packages/coverage/inorout.py:577: CoverageWarning: Module ioos_qc was previously imported, but not measured (module-not-measured); see https://coverage.readthedocs.io/en/7.14.1/messages.html#warning-module-not-measured
  self.warn(msg, slug="module-not-measured")


<ExitCode.OK: 0>

And it does look like we've improved testing coverage.
```shell
/home/aaron-mau/Code/gsoc/mau/ioos_qc/ioos_qc/qartod.py                            389     72    81%   61-62, 83-84, 143-144, 153-154, 219-220, 238-239, 302, 311-328, 339-340, 346-347, 352-353, 358-359, 366-368, 456-459, 875-876, 896, 960-961, 968, 1072-1089, 1117-1140
```
Lines 1024-1042 are no longer showing up!